# Midterm Project: Real-Time Classroom Monitoring

## 1. Scenario
A sensor in a university classroom sends a new reading **every minute**. We want to:
- receive the readings one by one, as they arrive,
- calculate live statistics,
- detect problems (too hot, too much CO₂, strange spikes),
- show the results while the data is coming in,
- save everything to files.

No real sensor is available, so a **Python generator** simulates it. It uses a fixed random seed, so the data is the same on every run.

**Data schema (one record per minute):**

| Field | Unit | Meaning |
|---|---|---|
| `timestamp` | date/time | when the reading was taken |
| `temperature` | °C | room temperature |
| `humidity` | % | relative humidity |
| `co2` | ppm | CO₂ level (rises when many people are in the room) |

In [ ]:
import time
from collections import deque

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

# Settings
N_READINGS = 180        # 180 minutes = 3 hours of data
WINDOW     = 10         # size of the rolling window (last 10 readings)
DELAY      = 0.05       # seconds to wait between readings (simulates real time)

TEMP_LIMIT = 27         # °C   -> "too hot"
CO2_LIMIT  = 1000       # ppm  -> "bad air, open a window"
SPIKE_JUMP = 3          # °C away from the moving average -> "spike"

## 2. Data generator
`sensor_stream()` produces **one reading at a time** with `yield`, like a real sensor. It simulates the following:
- a class from minute 30 to 120 makes the temperature and CO₂ go up,
- some readings are **missing** (`None`),
- some readings are random **spikes** (sensor errors).

In [ ]:
def sensor_stream(n=N_READINGS, seed=42):
    rng = np.random.default_rng(seed)
    start = pd.Timestamp("2026-10-05 09:00")
    temp, co2 = 21.0, 450.0

    for i in range(n):
        in_class = 30 <= i < 120                     # students are in the room
        temp += 0.1 * ((24.5 if in_class else 21.0) - temp) + rng.normal(0, 0.1)
        co2  += 0.1 * ((1200 if in_class else 450) - co2) + rng.normal(0, 10)

        reading = {
            "timestamp":   start + pd.Timedelta(minutes=i),
            "temperature": round(temp, 2),
            "humidity":    round(40 + rng.normal(0, 1), 1),
            "co2":         round(co2, 1),
        }
        if rng.random() < 0.03:                      # sensor glitch -> spike
            reading["temperature"] += 8
        if rng.random() < 0.03:                      # lost value
            reading["temperature"] = None
        yield reading


# Look at the first 3 readings
stream = sensor_stream()
for _ in range(3):
    print(next(stream))

## 3. Cleaning, indicators and anomaly detection
These functions process **one reading at a time**:
- `clean()`: if the temperature is missing, it uses the previous value.
- `indicators()`: uses NumPy on the last `WINDOW` readings to calculate the **moving average**, **rolling standard deviation** and **min/max**.
- `detect()`: simple rules that decide whether something is wrong.

In [ ]:
def clean(reading, last_temp):
    if reading["temperature"] is None:
        reading["temperature"] = last_temp
        reading["missing"] = True
    else:
        reading["missing"] = False
    return reading


def indicators(window):
    values = np.array(window)
    mean = values.mean()
    std  = values.std()
    return {
        "temp_avg": round(mean, 2),
        "temp_std": round(std, 2),
        "temp_min": values.min(),
        "temp_max": values.max(),
    }


def detect(reading, prev_window):
    alerts = []
    if reading["temperature"] > TEMP_LIMIT:
        alerts.append("HIGH_TEMP")
    if reading["co2"] > CO2_LIMIT:
        alerts.append("HIGH_CO2")
    # spike = sudden jump away from the recent moving average
    if len(prev_window) > 0 and abs(reading["temperature"] - np.mean(prev_window)) > SPIKE_JUMP:
        alerts.append("SPIKE")
    if reading["missing"]:
        alerts.append("MISSING_VALUE")
    return alerts

## 4. Real-time processing loop
Each new reading is cleaned, its indicators are calculated, and the rules are checked. A live table is shown every 10 readings. A `deque(maxlen=WINDOW)` always holds only the last 10 temperatures.

In [ ]:
window  = deque(maxlen=WINDOW)
results = []
last_temp = 21.0
alert_count = 0

for reading in sensor_stream():
    reading = clean(reading, last_temp)

    alerts = detect(reading, list(window))
    window.append(reading["temperature"])
    reading.update(indicators(window))
    reading["alerts"] = ", ".join(alerts)

    results.append(reading)
    last_temp = reading["temperature"]
    alert_count += len(alerts)

    # live display every 10 readings
    if len(results) % 10 == 0:
        clear_output(wait=True)
        print(f"Readings received: {len(results)}   |   alerts so far: {alert_count}")
        display(pd.DataFrame(results).tail(5))

    time.sleep(DELAY)

print("Stream finished.")

## 5. Save the results

In [ ]:
df = pd.DataFrame(results)
df.to_csv("results.csv", index=False)

alerts_df = df[df["alerts"] != ""]
alerts_df.to_csv("alerts.csv", index=False)

print(f"Saved {len(df)} readings to results.csv and {len(alerts_df)} alerts to alerts.csv")

## 6. Results table

In [ ]:
# Summary statistics of the stream
display(df[["temperature", "humidity", "co2", "temp_avg", "temp_std"]].describe().round(2))

# How many times each alert happened
alert_counts = df["alerts"].str.split(", ").explode().value_counts().drop("", errors="ignore")
display(alert_counts.rename("count").to_frame())

## 7. Visualization

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

# Temperature + moving average + spikes
ax1.plot(df["timestamp"], df["temperature"], color="lightsteelblue", label="temperature")
ax1.plot(df["timestamp"], df["temp_avg"], color="#2a78d6", linewidth=2, label=f"moving average ({WINDOW} min)")
spikes = df[df["alerts"].str.contains("SPIKE")]
ax1.scatter(spikes["timestamp"], spikes["temperature"], color="red", marker="x", s=80, zorder=3, label="spike")
ax1.axhline(TEMP_LIMIT, color="gray", linestyle="--", label=f"limit {TEMP_LIMIT} °C")
ax1.set_ylabel("Temperature (°C)")
ax1.set_title("Temperature with moving average and detected spikes")
ax1.legend(loc="upper left")

# CO2 + limit
ax2.plot(df["timestamp"], df["co2"], color="#eb6834", label="CO₂")
ax2.axhline(CO2_LIMIT, color="red", linestyle="--", label=f"limit {CO2_LIMIT} ppm")
ax2.set_ylabel("CO₂ (ppm)")
ax2.set_title("CO₂ level")
ax2.legend(loc="upper left")

ax2.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter("%H:%M"))
plt.tight_layout()
plt.savefig("results.png", dpi=120)
plt.show()

## 8. Conclusion
- The generator simulated a sensor, and every reading was processed **as soon as it arrived** (incremental processing).
- **Pandas** was used to build tables, save CSV files and summarise the results. **NumPy** was used for the rolling statistics.
- **Indicators:** moving average, rolling standard deviation, rolling min/max, and the alert count.
- **Rule-based detection:** high temperature, high CO₂, sudden spikes and missing values.
- **Findings:** during the class (about 09:30–11:00) the CO₂ goes above 1000 ppm, so the room needs more ventilation. Most temperature spikes were single bad readings, and the spike rule caught them.
- **Saved files:** `results.csv`, `alerts.csv`, `results.png`.